In [1]:
import json
# from model.speech_pipeline import SpeechPipeline
# from rag_pipeline.indexing.chunking import VADVTimeGapChunker
import os
# from rag_pipeline.indexing.embed import process_embedding_batches
from qdrant_client import QdrantClient
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from dotenv import load_dotenv
from rag_service.chain import RAGService
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv('/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/.env')

True

In [4]:
with open("/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/meta_data/youtube_processing_log.json", 'r') as f:
    downloaded_audio_data= json.load(f)

In [8]:
downloaded_audio_data['5b955ff576d7a656f689466daea6abe6']

{'url': 'https://www.youtube.com/watch?v=x1EfDuf-DoU',
 'url_id': '5b955ff576d7a656f689466daea6abe6',
 'status': 'completed',
 'steps_completed': ['processing_started',
  'download_started',
  'download_completed',
  'postprocess_started',
  'postprocess_completed',
  'all_steps_completed'],
 'created_at': '2025-10-07 12:24:56',
 'last_updated': '2025-10-07 12:29:11',
 'error': '\x1bERROR:\x1b unable to download video data: HTTP Error 403: Forbidden',
 'final_output_path': '/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data/audios/Dars e Quran  Surah Al-Kahf 1-8  Ashab e Kahf  Allama Syed Abdullah Tariq.mp3',
 'filename': 'Dars e Quran  Surah Al-Kahf 1-8  Ashab e Kahf  Allama Syed Abdullah Tariq'}

In [19]:
# speech_pipeline= SpeechPipeline(
#     vad_repo_or_dir='snakers4/silero-vad', 
#     vad_model_name='silero_vad', 
#     asr_model_name= "ai4bharat/indicconformer_stt_hi_hybrid_rnnt_large"
# )

# meta_data= {}
# for i in downloaded_audio_data:
#     file_output_path= downloaded_audio_data[i]['final_output_path']
#     if file_output_path and downloaded_audio_data[i]['status']== 'completed':
#         transcribed_vad_segments= speech_pipeline.process_audio_file(
#             filepath= file_output_path,
#             decoder= "ctc",
#             language_id= "hi",
#             vad_threshold= 0.3,
#             min_speech_duration_ms= 500,
#         )

#         output_file= file_output_path.replace('.mp3', '.json').replace("audios", "transcriptions")
#         with open(output_file, 'w') as f:
#             json.dump(transcribed_vad_segments, f, ensure_ascii= False, indent= 4)
            
#         meta_data[i]= downloaded_audio_data[i]
#         meta_data[i]['speech_pipeline_status']= 'completed'
#         meta_data[i]['speech_pipeline_output']= transcribed_vad_segments
#         meta_data[i]['speech_pipeline_output_path']= output_file

In [20]:
# data_dir= "/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data"
# file_dir= os.path.join(data_dir, 'transcriptions')
# file_name= "Dars e Quran  Surah Al-Kahf 1-8  Ashab e Kahf  Allama Syed Abdullah Tariq.json"
# file_path= os.path.join(file_dir, file_name)

# with open(file_path, 'r') as f:
#     vad_segments= json.load(f) 

In [21]:
# splitter= VADVTimeGapChunker(file_name= file_name, gap_threshold= 0.85, max_tokens= 500)
# chunks = splitter.split_documents(vad_segments)

In [22]:
# chunks[:2]

In [23]:
# chunks[0].metadata['token_count']

In [26]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
client = QdrantClient(path="/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/langchain_qdrant")
vector_store = QdrantVectorStore(
    client=client,
    collection_name="allama_rag_dev",
    embedding=embeddings,
)

In [27]:
# process_embedding_batches(chunks, vector_store.add_documents)

In [11]:
llm_model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

rag= RAGService(
    vector_store=vector_store,
    llm_model= llm_model
)

question= "what is salat or namaz that was given to prophet?"
res= rag.workflow(question)

In [12]:
print(res)

According to the provided transcript:

Salat was not merely "namaz" as commonly understood (the five daily prayers with specific rakats), but a "complete system" (mukammal nizam).

The literal meaning of "salat," according to the majority of linguists, is:
*   Dua (supplication)
*   Bestowing blessings
*   Honoring and showing respect
*   Helping someone grow, evolve, or thrive.

When it is said that Allah and His angels send "salat" upon the Prophet, it means they honor him, help him evolve, increase his respect, and pave his way.

In the context of the Prophet's responsibilities of prophethood, "qiyam" (standing) involved reflection and contemplation in the light of the Quran for a certain period, and "sajda" (prostration) meant bowing before Allah and living under His will. While "namaz" was also performed, it was emphasized that the Prophet was not made to pray all night, but to recite the Quran as much as was easy, to fulfill his responsibilities.

The text emphasizes that limitin

In [ ]:
#create qdrant client
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv
import os

load_dotenv('/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/.env')

qdrant_client = QdrantClient(
    url=os.getenv('QDRANT_ENDPOINT'), 
    api_key=os.getenv('QDRANT_API_KEY'),
)
qdrant_collection= "allama-rag-dev"

# qdrant_client.delete_collection(collection_name= qdrant_collection)

# qdrant_client.create_collection(
#     collection_name= qdrant_collection,
#     vectors_config= models.VectorParams(size= 3072, distance= models.Distance.COSINE),
)

True

In [12]:
import json 

with open("/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/meta_data/processed_files_logs.json", 'r') as f:
    downloaded_audio_data= json.load(f)
with open("/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/meta_data/processed_files_logs_2.json", 'r') as f:
    files_processed_data= json.load(f)

In [13]:
processed_files= {}

In [14]:
# for id in downloaded_audio_data:
#     if id not in files_processed_data:
#         processed_files[id]= downloaded_audio_data[id]
#     else:
#         processed_files[id]= files_processed_data[id]
#         if 'speech_pipeline_output' in processed_files[id]:
#             del processed_files[id]['speech_pipeline_output']

In [15]:
for id in downloaded_audio_data:
    if id in files_processed_data and 'filename' in downloaded_audio_data[id] and 'Ta-Ha' in downloaded_audio_data[id]['filename']:
        print(downloaded_audio_data[id])
        files_processed_data[id]= downloaded_audio_data[id]

{'url': 'https://www.youtube.com/watch?v=QSUIdan86rY', 'url_id': '008713ddbe19212c59ceba6b99ced6b1', 'status': 'completed', 'steps_completed': ['processing_started', 'download_started', 'download_completed', 'postprocess_started', 'postprocess_completed', 'all_steps_completed'], 'created_at': '2025-10-19 13:56:21', 'last_updated': '2025-10-19 13:57:40', 'error': None, 'final_output_path': '/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data/audios/Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB.mp3', 'filename': 'Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB', 'speech_pipeline_status': 'completed', 'speech_pipeline_output_path': '/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data/transcriptions/Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB.json', 'chunking_status': '

In [16]:
files_processed_data['008713ddbe19212c59ceba6b99ced6b1']

{'url': 'https://www.youtube.com/watch?v=QSUIdan86rY',
 'url_id': '008713ddbe19212c59ceba6b99ced6b1',
 'status': 'completed',
 'steps_completed': ['processing_started',
  'download_started',
  'download_completed',
  'postprocess_started',
  'postprocess_completed',
  'all_steps_completed'],
 'created_at': '2025-10-19 13:56:21',
 'last_updated': '2025-10-19 13:57:40',
 'error': None,
 'final_output_path': '/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data/audios/Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB.mp3',
 'filename': 'Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB',
 'speech_pipeline_status': 'completed',
 'speech_pipeline_output_path': '/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/processed_data/transcriptions/Darse Quran  Surah Ta-Ha (201-8)  The thought process of believers when DEEN becomes MAZHAB.json',


In [17]:
len(files_processed_data)

59

In [18]:
len(downloaded_audio_data), len(files_processed_data)

(43, 59)

In [19]:
with open("/Users/daniyalkhan/Documents/WORK-for-Compassion/projects/RAG-allama_audio/data/meta_data/processed_files_logs_2.json", 'w') as f:
    json.dump(files_processed_data, f, ensure_ascii= False, indent= 4)